# 07 — Integração WVS + V-Party

**Objetivo:** Construir a base analítica integrada entre os perfis afetivos individuais, os indicadores institucionais do V-Dem e as características partidárias do V-Party.

# 07 — Integração WVS + V-Party

## Objetivo

Este notebook tem como objetivo integrar a base multinível produzida na etapa anterior — contendo os indicadores individuais do World Values Survey (WVS) e os indicadores institucionais do Varieties of Democracy (V-Dem) — às informações partidárias provenientes do V-Party.

A unidade de análise permanecerá no nível do indivíduo. As características partidárias serão incorporadas por meio de chaves contextuais compatíveis com a base multinível, preservando integralmente os resultados produzidos nas etapas anteriores.

O produto final desta etapa será a base analítica consolidada:

`dados/integrados/base_analitica.parquet`

Esta etapa não recalcula fatores, perfis afetivos ou indicadores institucionais já produzidos. Ela utiliza exclusivamente arquivos persistidos em disco, em conformidade com a arquitetura modular, reprodutível e cumulativa do projeto.

Projeto carregado em:
C:\Users\tmaci\Documents\Doutorado_Populismo


## 1. Carregamento dos dados

In [2]:
# ============================================================
# 07 — Integração WVS + V-Party
# Configuração do ambiente
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Diretório raiz do projeto
# ------------------------------------------------------------------

ROOT = Path.cwd().parent

# adiciona a pasta scripts ao PYTHONPATH
sys.path.append(str(ROOT / "scripts"))

from project_tools import *

# ------------------------------------------------------------------
# Diretórios
# ------------------------------------------------------------------

DIR_DADOS = ROOT / "dados"
DIR_BRUTOS = DIR_DADOS / "brutos"
DIR_INTEGRADOS = DIR_DADOS / "integrados"
DIR_RESULTADOS = ROOT / "resultados"

# ------------------------------------------------------------------
# Arquivos
# ------------------------------------------------------------------

ARQ_BASE = DIR_INTEGRADOS / "base_multinivel.parquet"
ARQ_SAIDA = DIR_INTEGRADOS / "base_analitica.parquet"

print(f"Projeto : {ROOT}")
print(f"Entrada : {ARQ_BASE}")
print(f"Saída   : {ARQ_SAIDA}")

Projeto : C:\Users\tmaci\Documents\Doutorado_Populismo
Entrada : C:\Users\tmaci\Documents\Doutorado_Populismo\dados\integrados\base_multinivel.parquet
Saída   : C:\Users\tmaci\Documents\Doutorado_Populismo\dados\integrados\base_analitica.parquet


In [3]:
# ============================================================
# Célula 2 — Carregamento da base multinível
# ============================================================

assert ARQ_BASE.exists(), f"Arquivo não encontrado: {ARQ_BASE}"

base = pd.read_parquet(ARQ_BASE)

print("Base multinível carregada com sucesso.")
print("Dimensões:", base.shape)

display(base.head())
display(base.dtypes.value_counts())

Base multinível carregada com sucesso.
Dimensões: (39720, 53)


,country_text_id,year,national_pride,willing_to_fight,grandiosity,reject_different_race,reject_immigrants,reject_homosexuals,reject_other_religion,reject_different_language,...,imagined_subcommunity_proxy_w7,country_name,country_id,v2x_polyarchy,v2x_libdem,v2x_partipdem,v2x_delibdem,v2x_egaldem,v2x_api,v2x_mpi
0,ARG,2013,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.413657,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645
1,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.342130,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645
2,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.504398,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645
3,ARG,2013,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.453241,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645
4,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.335417,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645


float64    50
object      2
int64       1
Name: count, dtype: int64

In [6]:
# ============================================================
# Célula 3 — Localização da base V-Party
# ============================================================

ARQ_VPARTY = next(DIR_BRUTOS.rglob("V-Dem-CPD-Party-V2*.csv"))

print(ARQ_VPARTY)

C:\Users\tmaci\Documents\Doutorado_Populismo\dados\brutos\VPARTY\V-Dem-CPD-Party-V2.csv


## 2. Preparação

In [7]:
# ============================================================
# Célula 4 — Leitura inicial do V-Party
# ============================================================

vparty = pd.read_csv(ARQ_VPARTY, low_memory=False)

print("V-Party carregado com sucesso.")
print("Dimensões:", vparty.shape)

display(vparty.head())
display(vparty.dtypes.value_counts())

print("\nPrimeiras 40 colunas:")
for col in vparty.columns[:40]:
    print(col)

V-Party carregado com sucesso.
Dimensões: (11898, 384)


,v2paenname,v2paorname,v2pashname,v2paid,pf_party_id,party_gaps,pf_url,country_name,histname,country_id,...,ep_people_vs_elite,ep_v7_lib_cons_saliency,ep_type_populism,ep_type_populist_values,ep_v8_popul_rhetoric,ep_v9_popul_saliency,e_regiongeo,e_regionpol,e_regionpol_6C,GPS_ID
0,Party of the Democratic Revolution,NaN,PRD,216,216.0,NaN,https://partyfacts.herokuapp.com/data/partycod...,Mexico,United Mexican States,3,...,NaN,NaN,NaN,NaN,NaN,NaN,17,2,2,NaN
1,Party of the Democratic Revolution,NaN,PRD,216,216.0,NaN,https://partyfacts.herokuapp.com/data/partycod...,Mexico,United Mexican States,3,...,NaN,NaN,NaN,NaN,NaN,NaN,17,2,2,NaN
2,Party of the Democratic Revolution,NaN,PRD,216,216.0,NaN,https://partyfacts.herokuapp.com/data/partycod...,Mexico,United Mexican States,3,...,NaN,NaN,NaN,NaN,NaN,NaN,17,2,2,NaN
3,Party of the Democratic Revolution,NaN,PRD,216,216.0,NaN,https://partyfacts.herokuapp.com/data/partycod...,Mexico,United Mexican States,3,...,NaN,NaN,NaN,NaN,NaN,NaN,17,2,2,NaN
4,Party of the Democratic Revolution,NaN,PRD,216,216.0,NaN,https://partyfacts.herokuapp.com/data/partycod...,Mexico,United Mexican States,3,...,NaN,NaN,NaN,NaN,NaN,NaN,17,2,2,NaN


float64    364
object      10
int64       10
Name: count, dtype: int64


Primeiras 40 colunas:
v2paenname
v2paorname
v2pashname
v2paid
pf_party_id
party_gaps
pf_url
country_name
histname
country_id
country_text_id
year
historical_date
codingstart
gapstart
gapend
codingend
project
COWcode
gap_index
v2xpa_antiplural
v2xpa_antiplural_codelow
v2xpa_antiplural_codehigh
v2xpa_popul
v2xpa_popul_codelow
v2xpa_popul_codehigh
v2paseatshare
v2panumbseat
v2patotalseat
v2pavote
v2paallian
v2panaallian
v2pavallian
v2panoallian
v2paelcont
v2paelcont_nr
v2pagovsup
v2papariah
v2papariah_codelow
v2papariah_codehigh


In [9]:
# ============================================================
# Célula 6 — Estrutura da chave do V-Party
# ============================================================

print("Observações:", len(vparty))

print("\nPaíses:", vparty["country_text_id"].nunique())
print("Anos:", vparty["year"].nunique())

print("\nPartidos por país-ano:")

(
    vparty
    .groupby(["country_text_id", "year"])
    .size()
    .describe()
)

Observações: 11898

Países: 178
Anos: 120

Partidos por país-ano:


count    3151.000000
mean        3.775944
std         2.480900
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        29.000000
dtype: float64

In [10]:
# ============================================================
# Célula 7 — Seleção das variáveis substantivas do V-Party
# ============================================================

# Variáveis de identificação
id_vars = [
    "country_text_id",
    "country_name",
    "year",
    "historical_date",
    "v2paid",
    "v2paenname",
    "v2paorname",
    "v2pashname",
]

# Peso político-eleitoral
peso_vars = [
    "v2pavote",
    "v2paseatshare",
    "v2panumbseat",
    "v2patotalseat",
    "v2pagovsup",
]

# Organização partidária
organizacao_vars = [
    "v2paelcont",
    "v2palocoff",
    "v2paopresp",
    "v2paclient",
]

# Populismo
populismo_vars = [
    "v2xpa_popul",
    "v2xpa_antiplural",
    "v2papeople",
    "v2paanteli",
    "v2paplur",
    "v2paculsup",
]

# Ideologia
ideologia_vars = [
    "v2pariglef",
]

# Variáveis alternativas (Expert Party Survey)
expert_vars = [
    "ep_people_vs_elite",
    "ep_antielite_salience",
    "ep_v8_popul_rhetoric",
    "ep_v9_popul_saliency",
    "ep_galtan",
]

# Lista final
vars_vparty = (
    id_vars
    + peso_vars
    + organizacao_vars
    + populismo_vars
    + ideologia_vars
    + expert_vars
)

# Mantém apenas as existentes
vars_existentes = [v for v in vars_vparty if v in vparty.columns]

vparty_limpo = vparty[vars_existentes].copy()

print(f"Variáveis selecionadas: {len(vars_existentes)}")
print(f"Dimensão da base: {vparty_limpo.shape}")

display(vparty_limpo.head())

Variáveis selecionadas: 29
Dimensão da base: (11898, 29)


,country_text_id,country_name,year,historical_date,v2paid,v2paenname,v2paorname,v2pashname,v2pavote,v2paseatshare,...,v2papeople,v2paanteli,v2paplur,v2paculsup,v2pariglef,ep_people_vs_elite,ep_antielite_salience,ep_v8_popul_rhetoric,ep_v9_popul_saliency,ep_galtan
0,MEX,Mexico,1991,1991-08-18,216,Party of the Democratic Revolution,NaN,PRD,8.3,8.2,...,0.811,2.020,0.985,1.115,-1.531,NaN,NaN,NaN,NaN,NaN
1,MEX,Mexico,1994,1994-08-21,216,Party of the Democratic Revolution,NaN,PRD,16.7,14.2,...,0.811,1.878,0.908,1.115,-1.531,NaN,NaN,NaN,NaN,NaN
2,MEX,Mexico,1997,1997-07-06,216,Party of the Democratic Revolution,NaN,PRD,25.7,25.0,...,0.740,1.371,1.133,1.115,-1.531,NaN,NaN,NaN,NaN,NaN
3,MEX,Mexico,2000,2000-07-02,216,Party of the Democratic Revolution,NaN,PRD,18.7,13.0,...,0.740,1.464,0.902,1.115,-1.531,NaN,NaN,NaN,NaN,NaN
4,MEX,Mexico,2003,2003-06-06,216,Party of the Democratic Revolution,NaN,PRD,18.2,19.4,...,0.816,1.366,0.660,0.710,-1.531,NaN,NaN,NaN,NaN,NaN


In [11]:
# ============================================================
#Inventário das variáveis do V-Party
# ============================================================

colunas = sorted(vparty.columns)

print(f"Total de variáveis: {len(colunas)}\n")

for c in colunas:
    print(c)

Total de variáveis: 384

CHES_ID
COWcode
GPS_ID
codingend
codingstart
country_id
country_name
country_text_id
e_regiongeo
e_regionpol
e_regionpol_6C
ep_antielite_salience
ep_corrupt_salience
ep_galtan
ep_galtan_salience
ep_members_vs_leadership
ep_people_vs_elite
ep_type_populism
ep_type_populist_values
ep_v6_lib_cons
ep_v7_lib_cons_saliency
ep_v8_popul_rhetoric
ep_v9_popul_saliency
gap_index
gapend
gapstart
histname
historical_date
party_gaps
pf_party_id
pf_url
project
v2paactcom
v2paactcom_codehigh
v2paactcom_codelow
v2paactcom_mean
v2paactcom_nr
v2paactcom_ord
v2paactcom_ord_codehigh
v2paactcom_ord_codelow
v2paactcom_osp
v2paactcom_osp_codehigh
v2paactcom_osp_codelow
v2paactcom_osp_sd
v2paactcom_sd
v2paallian
v2paanteli
v2paanteli_codehigh
v2paanteli_codelow
v2paanteli_mean
v2paanteli_nr
v2paanteli_ord
v2paanteli_ord_codehigh
v2paanteli_ord_codelow
v2paanteli_osp
v2paanteli_osp_codehigh
v2paanteli_osp_codelow
v2paanteli_osp_sd
v2paanteli_sd
v2paclient
v2paclient_codehigh
v2paclient_

In [12]:
# ============================================================
# Diagnóstico da base V-Party
# ============================================================

diagnostico = (
    vparty_limpo
    .isna()
    .sum()
    .to_frame("missing")
)

diagnostico["%"] = (
    diagnostico["missing"]
    / len(vparty_limpo)
    * 100
).round(2)

diagnostico = diagnostico.sort_values("%", ascending=False)

display(diagnostico)

print("\nTotal de observações:", len(vparty_limpo))
print("Total de partidos:", vparty_limpo["v2paid"].nunique())
print("Total de países:", vparty_limpo["country_text_id"].nunique())
print("Período:",
      vparty_limpo["year"].min(),
      "-",
      vparty_limpo["year"].max())

,missing,%
ep_people_vs_elite,11807,99.24
ep_antielite_salience,11769,98.92
ep_galtan,11604,97.53
ep_v9_popul_saliency,11379,95.64
ep_v8_popul_rhetoric,11377,95.62
v2paorname,10142,85.24
v2palocoff,5640,47.40
v2pagovsup,5617,47.21
v2paclient,5600,47.07
v2paopresp,5596,47.03



Total de observações: 11898
Total de partidos: 3467
Total de países: 178
Período: 1900 - 2019


In [13]:
# ============================================================
# Verificação da chave do V-Party
# ============================================================

duplicatas = (
    vparty_limpo
    .duplicated(
        subset=["country_text_id", "year", "v2paid"],
        keep=False
    )
)

print("Registros duplicados:", duplicatas.sum())

if duplicatas.any():
    display(
        vparty_limpo.loc[duplicatas]
        .sort_values(["country_text_id", "year", "v2paid"])
    )

Registros duplicados: 0


In [14]:
vparty_limpo.to_parquet(
    DIR_INTEGRADOS / "vparty_limpo.parquet",
    index=False
)

## 3. Análise

In [15]:
# ============================================================
# Construção do peso partidário
# ============================================================

vparty_limpo["peso_partido"] = (
    vparty_limpo["v2paseatshare"]
    .fillna(vparty_limpo["v2pavote"])
)

print(vparty_limpo["peso_partido"].describe())

print(
    "\nCasos sem peso:",
    vparty_limpo["peso_partido"].isna().sum()
)

count    11043.000000
mean        26.947786
std         25.600794
min          0.000000
25%          7.700000
50%         17.400000
75%         39.450000
max        100.000000
Name: peso_partido, dtype: float64

Casos sem peso: 855


In [16]:
vparty_limpo["peso_partido"] = (
    vparty_limpo["v2paseatshare"]
        .fillna(vparty_limpo["v2pavote"])
        .fillna(0)
)

In [17]:
def media_ponderada(grupo, valor, peso="peso_partido"):
    dados = grupo[[valor, peso]].dropna()

    if dados.empty:
        return np.nan

    if dados[peso].sum() == 0:
        return dados[valor].mean()

    return np.average(
        dados[valor],
        weights=dados[peso]
    )

In [18]:
# ============================================================
# Construção da base do sistema partidário
# ============================================================

chaves = ["country_text_id", "year"]

vparty_agregado = (
    vparty_limpo
    .groupby(chaves)
    .apply(
        lambda g: pd.Series({

            # Estrutura do sistema
            "n_partidos": g["v2paid"].nunique(),

            # Peso eleitoral
            "participacao_media": media_ponderada(g, "v2paseatshare"),
            "voto_medio": media_ponderada(g, "v2pavote"),

            # Organização
            "organizacao_media": media_ponderada(g, "v2paelcont"),
            "capilaridade_media": media_ponderada(g, "v2palocoff"),
            "oposicao_media": media_ponderada(g, "v2paopresp"),
            "clientelismo_medio": media_ponderada(g, "v2paclient"),

            # Populismo
            "populismo_medio": media_ponderada(g, "v2xpa_popul"),
            "populismo_max": g["v2xpa_popul"].max(),

            "antipluralismo_medio": media_ponderada(g, "v2xpa_antiplural"),
            "antielitismo_medio": media_ponderada(g, "v2paanteli"),
            "povo_vs_elite_medio": media_ponderada(g, "v2papeople"),
            "pluralismo_medio": media_ponderada(g, "v2paplur"),
            "culto_lider_medio": media_ponderada(g, "v2paculsup"),

            # Ideologia
            "direita_esquerda_media": media_ponderada(g, "v2pariglef")

        })
    )
    .reset_index()
)

print(vparty_agregado.shape)

display(vparty_agregado.head())

(3151, 17)


C:\Users\tmaci\AppData\Local\Temp\ipykernel_15992\1403249247.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,country_text_id,year,n_partidos,participacao_media,voto_medio,organizacao_media,capilaridade_media,oposicao_media,clientelismo_medio,populismo_medio,populismo_max,antipluralismo_medio,antielitismo_medio,povo_vs_elite_medio,pluralismo_medio,culto_lider_medio,direita_esquerda_media
0,AFG,1965,1.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,1969,1.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,1988,1.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,2005,1.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,2010,1.0,100.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# ============================================================
# Número efetivo de partidos (Laakso & Taagepera)
# ============================================================

def numero_efetivo_partidos(grupo):
    pesos = grupo["peso_partido"].fillna(0).astype(float)

    if pesos.sum() == 0:
        return np.nan

    p = pesos / pesos.sum()

    return 1 / np.sum(p**2)


enp = (
    vparty_limpo
    .groupby(["country_text_id", "year"])
    .apply(numero_efetivo_partidos)
    .rename("numero_efetivo_partidos")
    .reset_index()
)

vparty_agregado = vparty_agregado.merge(
    enp,
    on=["country_text_id", "year"],
    how="left"
)

display(vparty_agregado[
    ["country_text_id",
     "year",
     "n_partidos",
     "numero_efetivo_partidos"]
].head(15))

C:\Users\tmaci\AppData\Local\Temp\ipykernel_15992\3807531834.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(numero_efetivo_partidos)


,country_text_id,year,n_partidos,numero_efetivo_partidos
0,AFG,1965,1.0,1.000000
1,AFG,1969,1.0,1.000000
2,AFG,1988,1.0,1.000000
3,AFG,2005,1.0,1.000000
4,AFG,2010,1.0,1.000000
5,AGO,1975,1.0,NaN
6,AGO,1980,1.0,1.000000
7,AGO,1986,1.0,1.000000
8,AGO,1992,2.0,1.838423
9,AGO,2008,2.0,1.167021


In [20]:
# ============================================================
# Funções para indicadores do sistema partidário
# ============================================================

def numero_efetivo_partidos(grupo, peso="peso_partido"):
    pesos = grupo[peso].fillna(0).astype(float)

    if pesos.sum() == 0:
        return np.nan

    p = pesos / pesos.sum()

    return 1 / np.sum(p**2)


def concentracao(grupo, peso="peso_partido"):
    pesos = grupo[peso].fillna(0)

    if pesos.sum() == 0:
        return np.nan

    return pesos.max() / pesos.sum()


def media_ponderada(grupo, valor, peso="peso_partido"):
    dados = grupo[[valor, peso]].dropna()

    if dados.empty:
        return np.nan

    if dados[peso].sum() == 0:
        return dados[valor].mean()

    return np.average(
        dados[valor],
        weights=dados[peso]
    )

In [21]:
# ============================================================
# Concentração do sistema partidário
# ============================================================

concentracao = (
    vparty_limpo
    .groupby(["country_text_id", "year"])
    .apply(concentracao)
    .rename("concentracao_partidaria")
    .reset_index()
)

vparty_agregado = (
    vparty_agregado
    .merge(
        concentracao,
        on=["country_text_id", "year"],
        how="left"
    )
)

display(
    vparty_agregado[
        [
            "country_text_id",
            "year",
            "n_partidos",
            "numero_efetivo_partidos",
            "concentracao_partidaria"
        ]
    ].head(15)
)

C:\Users\tmaci\AppData\Local\Temp\ipykernel_15992\2252177753.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(concentracao)


,country_text_id,year,n_partidos,numero_efetivo_partidos,concentracao_partidaria
0,AFG,1965,1.0,1.000000,1.000000
1,AFG,1969,1.0,1.000000,1.000000
2,AFG,1988,1.0,1.000000,1.000000
3,AFG,2005,1.0,1.000000,1.000000
4,AFG,2010,1.0,1.000000,1.000000
5,AGO,1975,1.0,NaN,NaN
6,AGO,1980,1.0,1.000000,1.000000
7,AGO,1986,1.0,1.000000,1.000000
8,AGO,1992,2.0,1.838423,0.648230
9,AGO,2008,2.0,1.167021,0.922423


In [23]:
# ============================================================
# Indicadores contínuos do sistema partidário
# ============================================================

def media_ponderada_escore(grupo, valor, peso="peso_partido"):
    dados = grupo[[valor, peso]].dropna()

    if dados.empty:
        return np.nan

    if dados[peso].sum() == 0:
        return np.nan

    return np.average(dados[valor], weights=dados[peso])


DIMENSOES = {
    "populismo": "v2xpa_popul",
    "antipluralismo": "v2xpa_antiplural",
    "antielitismo": "v2paanteli",
    "peoplecentrism": "v2papeople",
    "pluralismo": "v2paplur",
    "culto_lider": "v2paculsup",
    "organizacao": "v2paelcont",
    "clientelismo": "v2paclient",
    "ideologia": "v2pariglef",
}


def indicadores_continuos(grupo):
    saida = {}

    for nome, var in DIMENSOES.items():
        serie = grupo[var].dropna()

        saida[f"{nome}_ponderado"] = media_ponderada_escore(grupo, var)
        saida[f"{nome}_max"] = serie.max() if not serie.empty else np.nan
        saida[f"{nome}_sd"] = serie.std() if len(serie) > 1 else np.nan
        saida[f"{nome}_range"] = (
            serie.max() - serie.min()
            if not serie.empty
            else np.nan
        )

    return pd.Series(saida)


indicadores_sistema = (
    vparty_limpo
    .groupby(["country_text_id", "year"])
    .apply(indicadores_continuos)
    .reset_index()
)

# Remove indicadores antigos que possam ter sido criados antes da refatoração
cols_remover = [
    col for col in vparty_agregado.columns
    if (
        col.startswith("populismo_")
        or col.startswith("antipluralismo_")
        or col.startswith("antielitismo_")
        or col.startswith("peoplecentrism_")
        or col.startswith("pluralismo_")
        or col.startswith("culto_lider_")
        or col.startswith("organizacao_")
        or col.startswith("clientelismo_")
        or col.startswith("ideologia_")
    )
]

vparty_agregado = (
    vparty_agregado
    .drop(columns=cols_remover, errors="ignore")
    .merge(
        indicadores_sistema,
        on=["country_text_id", "year"],
        how="left"
    )
)

display(vparty_agregado.head())

C:\Users\tmaci\AppData\Local\Temp\ipykernel_15992\2586973764.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(indicadores_continuos)


,country_text_id,year,n_partidos,participacao_media,voto_medio,capilaridade_media,oposicao_media,povo_vs_elite_medio,direita_esquerda_media,numero_efetivo_partidos,...,organizacao_sd,organizacao_range,clientelismo_ponderado,clientelismo_max,clientelismo_sd,clientelismo_range,ideologia_ponderado,ideologia_max,ideologia_sd,ideologia_range
0,AFG,1965,1.0,100.0,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,1969,1.0,100.0,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,1988,1.0,100.0,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,2005,1.0,100.0,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,2010,1.0,100.0,100.0,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
display(
    vparty_agregado.loc[
        (vparty_agregado["country_text_id"] == "AFG") &
        (vparty_agregado["year"].isin([1965, 1969, 1988, 2005, 2010])),
        [
            "country_text_id",
            "year",
            "populismo_ponderado",
            "populismo_max",
            "populismo_sd",
            "populismo_range"
        ]
    ]
)

,country_text_id,year,populismo_ponderado,populismo_max,populismo_sd,populismo_range
0,AFG,1965,NaN,NaN,NaN,NaN
1,AFG,1969,NaN,NaN,NaN,NaN
2,AFG,1988,NaN,NaN,NaN,NaN
3,AFG,2005,NaN,NaN,NaN,NaN
4,AFG,2010,NaN,NaN,NaN,NaN


In [25]:
# ============================================================
# Recorte do V-Party para o universo analítico da tese
# ============================================================

# Países presentes na base multinível (WVS + V-Dem)
paises_base = set(base["country_text_id"].dropna().unique())

vparty_americas = (
    vparty_limpo
    .loc[vparty_limpo["country_text_id"].isin(paises_base)]
    .copy()
)

print(f"Observações (mundo):   {len(vparty_limpo):,}")
print(f"Observações (Américas): {len(vparty_americas):,}")

print(f"Países (mundo):   {vparty_limpo['country_text_id'].nunique()}")
print(f"Países (Américas): {vparty_americas['country_text_id'].nunique()}")

# Persistência da base recortada
vparty_americas.to_parquet(
    DIR_INTEGRADOS / "vparty_americas.parquet",
    index=False
)

Observações (mundo):   11,898
Observações (Américas): 1,638
Países (mundo):   178
Países (Américas): 16


In [26]:
# A partir deste ponto, todas as agregações serão realizadas
# exclusivamente sobre o recorte das Américas.

vparty = vparty_americas.copy()

In [27]:
# ============================================================
# Inicialização da base agregada do sistema partidário
# ============================================================

chaves = ["country_text_id", "year"]

vparty_agregado = (
    vparty[chaves]
    .drop_duplicates()
    .sort_values(chaves)
    .reset_index(drop=True)
)

print(vparty_agregado.shape)

display(vparty_agregado.head())

(437, 2)


,country_text_id,year
0,ARG,1912
1,ARG,1914
2,ARG,1916
3,ARG,1918
4,ARG,1920


In [28]:
# ============================================================
# Estrutura do sistema partidário
# ============================================================

def numero_efetivo_partidos(grupo, peso="peso_partido"):
    pesos = grupo[peso].fillna(0).astype(float)

    if pesos.sum() == 0:
        return np.nan

    p = pesos / pesos.sum()
    return 1 / np.sum(p**2)


def concentracao_partidaria(grupo, peso="peso_partido"):
    pesos = grupo[peso].fillna(0).astype(float)

    if pesos.sum() == 0:
        return np.nan

    return pesos.max() / pesos.sum()


estrutura_sistema = (
    vparty
    .groupby(chaves)
    .apply(
        lambda g: pd.Series({
            "n_partidos": g["v2paid"].nunique(),
            "numero_efetivo_partidos": numero_efetivo_partidos(g),
            "concentracao_partidaria": concentracao_partidaria(g),
        })
    )
    .reset_index()
)

vparty_agregado = vparty_agregado.merge(
    estrutura_sistema,
    on=chaves,
    how="left"
)

display(vparty_agregado.head())

C:\Users\tmaci\AppData\Local\Temp\ipykernel_15992\3353797546.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,country_text_id,year,n_partidos,numero_efetivo_partidos,concentracao_partidaria
0,ARG,1912,6.0,4.524729,0.317872
1,ARG,1914,3.0,2.580550,0.451550
2,ARG,1916,4.0,2.765375,0.494609
3,ARG,1918,5.0,2.797855,0.543655
4,ARG,1920,4.0,2.089838,0.661692


In [29]:
# ============================================================
# Indicadores contínuos do campo partidário
# ============================================================

def media_ponderada_escore(grupo, valor, peso="peso_partido"):
    dados = grupo[[valor, peso]].dropna()

    if dados.empty or dados[peso].sum() == 0:
        return np.nan

    return np.average(dados[valor], weights=dados[peso])


DIMENSOES = {
    "populismo": "v2xpa_popul",
    "antipluralismo": "v2xpa_antiplural",
    "antielitismo": "v2paanteli",
    "peoplecentrism": "v2papeople",
    "pluralismo": "v2paplur",
    "culto_lider": "v2paculsup",
    "organizacao": "v2paelcont",
    "capilaridade": "v2palocoff",
    "oposicao": "v2paopresp",
    "clientelismo": "v2paclient",
    "ideologia": "v2pariglef",
}


def indicadores_continuos(grupo):
    saida = {}

    for nome, var in DIMENSOES.items():
        serie = grupo[var].dropna()

        saida[f"{nome}_ponderado"] = media_ponderada_escore(grupo, var)
        saida[f"{nome}_max"] = serie.max() if not serie.empty else np.nan
        saida[f"{nome}_sd"] = serie.std() if len(serie) > 1 else np.nan
        saida[f"{nome}_range"] = (
            serie.max() - serie.min()
            if not serie.empty
            else np.nan
        )

    return pd.Series(saida)


indicadores_sistema = (
    vparty
    .groupby(chaves)
    .apply(indicadores_continuos)
    .reset_index()
)

vparty_agregado = vparty_agregado.merge(
    indicadores_sistema,
    on=chaves,
    how="left"
)

display(vparty_agregado.head())

C:\Users\tmaci\AppData\Local\Temp\ipykernel_15992\436040537.py:50: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(indicadores_continuos)


,country_text_id,year,n_partidos,numero_efetivo_partidos,concentracao_partidaria,populismo_ponderado,populismo_max,populismo_sd,populismo_range,antipluralismo_ponderado,...,oposicao_sd,oposicao_range,clientelismo_ponderado,clientelismo_max,clientelismo_sd,clientelismo_range,ideologia_ponderado,ideologia_max,ideologia_sd,ideologia_range
0,ARG,1912,6.0,4.524729,0.317872,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ARG,1914,3.0,2.580550,0.451550,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ARG,1916,4.0,2.765375,0.494609,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ARG,1918,5.0,2.797855,0.543655,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ARG,1920,4.0,2.089838,0.661692,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# ============================================================
# Polarização ideológica ponderada
# ============================================================

def desvio_ponderado(grupo, valor, peso="peso_partido"):
    dados = grupo[[valor, peso]].dropna()

    if dados.empty or dados[peso].sum() == 0:
        return np.nan

    media = np.average(dados[valor], weights=dados[peso])

    variancia = np.average(
        (dados[valor] - media) ** 2,
        weights=dados[peso]
    )

    return np.sqrt(variancia)


polarizacao = (
    vparty
    .groupby(chaves)
    .apply(
        lambda g: pd.Series({
            "polarizacao_ideologica": desvio_ponderado(g, "v2pariglef"),
            "heterogeneidade_populista": desvio_ponderado(g, "v2xpa_popul"),
        })
    )
    .reset_index()
)

vparty_agregado = vparty_agregado.merge(
    polarizacao,
    on=chaves,
    how="left"
)

display(
    vparty_agregado[
        [
            "country_text_id",
            "year",
            "ideologia_ponderado",
            "ideologia_range",
            "polarizacao_ideologica",
            "populismo_ponderado",
            "populismo_range",
            "heterogeneidade_populista"
        ]
    ].dropna().head(10)
)

C:\Users\tmaci\AppData\Local\Temp\ipykernel_15992\1694653717.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,country_text_id,year,ideologia_ponderado,ideologia_range,polarizacao_ideologica,populismo_ponderado,populismo_range,heterogeneidade_populista
25,ARG,1973,-0.659103,1.405,0.343774,0.755291,0.494,0.224953
26,ARG,1983,-0.479566,0.055,0.027422,0.659088,0.184,0.091740
27,ARG,1985,-0.766685,0.854,0.179758,0.610460,0.150,0.073452
28,ARG,1987,-0.540483,3.957,0.600673,0.573686,0.588,0.114001
29,ARG,1989,-0.311018,3.152,0.685374,0.539421,0.589,0.159288
30,ARG,1991,0.573834,2.319,0.529918,0.350265,0.341,0.071509
31,ARG,1993,0.794979,1.278,0.625641,0.299474,0.074,0.036226
32,ARG,1995,0.547289,2.375,0.773834,0.317163,0.342,0.096582
33,ARG,1997,0.393901,2.372,0.908545,0.355297,0.329,0.118697
34,ARG,1999,0.420783,3.119,0.718921,0.354907,0.390,0.086140


In [31]:
# ============================================================
# Diagnóstico de cobertura dos indicadores partidários
# ============================================================

def cobertura_variavel(grupo, variavel, peso="peso_partido"):
    dados = grupo[[variavel, peso]].copy()

    peso_total = dados[peso].fillna(0).sum()

    if peso_total == 0:
        return np.nan

    peso_valido = (
        dados
        .loc[dados[variavel].notna(), peso]
        .fillna(0)
        .sum()
    )

    return peso_valido / peso_total


cobertura_sistema = (
    vparty
    .groupby(chaves)
    .apply(
        lambda g: pd.Series({
            "cobertura_populismo": cobertura_variavel(g, "v2xpa_popul"),
            "cobertura_antipluralismo": cobertura_variavel(g, "v2xpa_antiplural"),
            "cobertura_ideologia": cobertura_variavel(g, "v2pariglef"),
            "cobertura_organizacao": cobertura_variavel(g, "v2paelcont"),
            "cobertura_clientelismo": cobertura_variavel(g, "v2paclient"),
            "n_partidos_sistema": g["v2paid"].nunique(),
            "n_partidos_com_populismo": g.loc[g["v2xpa_popul"].notna(), "v2paid"].nunique(),
            "n_partidos_com_ideologia": g.loc[g["v2pariglef"].notna(), "v2paid"].nunique(),
        })
    )
    .reset_index()
)

vparty_agregado = vparty_agregado.merge(
    cobertura_sistema,
    on=chaves,
    how="left"
)

display(
    vparty_agregado[
        [
            "country_text_id",
            "year",
            "n_partidos_sistema",
            "n_partidos_com_populismo",
            "cobertura_populismo",
            "n_partidos_com_ideologia",
            "cobertura_ideologia"
        ]
    ].dropna().head(15)
)

C:\Users\tmaci\AppData\Local\Temp\ipykernel_15992\431422774.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,country_text_id,year,n_partidos_sistema,n_partidos_com_populismo,cobertura_populismo,n_partidos_com_ideologia,cobertura_ideologia
0,ARG,1912,6.0,0.0,0.0,0.0,0.0
1,ARG,1914,3.0,0.0,0.0,0.0,0.0
2,ARG,1916,4.0,0.0,0.0,0.0,0.0
3,ARG,1918,5.0,0.0,0.0,0.0,0.0
4,ARG,1920,4.0,0.0,0.0,0.0,0.0
5,ARG,1922,5.0,0.0,0.0,0.0,0.0
6,ARG,1924,6.0,0.0,0.0,0.0,0.0
7,ARG,1926,4.0,0.0,0.0,0.0,0.0
8,ARG,1928,4.0,0.0,0.0,0.0,0.0
9,ARG,1930,5.0,0.0,0.0,0.0,0.0


In [32]:
# ============================================================
# Cobertura temporal do V-Party no universo da tese
# ============================================================

cobertura_ano = (
    vparty_agregado
    .groupby("year")
    .agg(
        paises=("country_text_id", "nunique"),
        cobertura_populismo=("cobertura_populismo", "mean"),
        cobertura_ideologia=("cobertura_ideologia", "mean")
    )
    .reset_index()
)

display(cobertura_ano.tail(25))

,year,paises,cobertura_populismo,cobertura_ideologia
86,1995,5,1.000000,1.000000
87,1996,3,1.000000,1.000000
88,1997,6,0.950758,0.950758
89,1998,5,1.000000,1.000000
90,1999,3,0.890143,0.890143
91,2000,7,1.000000,1.000000
92,2001,5,0.967960,0.967960
93,2002,6,0.998062,0.998062
94,2003,3,1.000000,1.000000
95,2004,3,1.000000,1.000000


In [33]:
print(
    vparty_agregado[
        ["cobertura_populismo",
         "cobertura_ideologia"]
    ].describe()
)

       cobertura_populismo  cobertura_ideologia
count           437.000000           437.000000
mean              0.483738             0.483738
std               0.495131             0.495131
min               0.000000             0.000000
25%               0.000000             0.000000
50%               0.000000             0.000000
75%               1.000000             1.000000
max               1.000000             1.000000


In [34]:
# ============================================================
# Diagnóstico de cobertura da integração
# ============================================================

diagnostico_merge = (
    base[
        ["country_text_id", "year"]
    ]
    .drop_duplicates()
    .merge(
        vparty_agregado[["country_text_id", "year"]],
        on=["country_text_id", "year"],
        how="left",
        indicator=True
    )
)

print(diagnostico_merge["_merge"].value_counts())

print("\nCobertura da base multinível:")

print(
    (
        diagnostico_merge["_merge"]
        == "both"
    ).mean()
)

_merge
left_only     17
both           9
right_only     0
Name: count, dtype: int64

Cobertura da base multinível:
0.34615384615384615


In [35]:
print(
    base[
        ["country_text_id", "year"]
    ]
    .drop_duplicates()
    .sort_values(["country_text_id", "year"])
)

      country_text_id  year
0                 ARG  2013
1030              ARG  2017
2033              BOL  2017
4100              BRA  2014
5586              BRA  2018
7348              CAN  2020
11366             CHL  2012
12366             CHL  2018
13366             COL  2012
14878             COL  2018
16398             ECU  2013
17600             ECU  2018
18800             GTM  2020
20029             HTI  2016
22025             MEX  2012
24025             MEX  2018
25766             NIC  2020
26966             PER  2012
28176             PER  2018
29576             PRI  2018
30703             TTO  2010
31702             URY  2011
32702             URY  2022
33702             USA  2011
35934             USA  2017
38530             VEN  2021


In [36]:
# ============================================================
# Harmonização temporal WVS × V-Party
# ============================================================

# Contextos existentes na base multinível
contextos = (
    base[
        ["country_text_id", "year"]
    ]
    .drop_duplicates()
    .rename(columns={"year": "year_wvs"})
    .sort_values(["country_text_id", "year_wvs"])
    .reset_index(drop=True)
)

# Contextos disponíveis no V-Party
eleicoes = (
    vparty_agregado[
        ["country_text_id", "year"]
    ]
    .rename(columns={"year": "year_eleicao"})
    .sort_values(["country_text_id", "year_eleicao"])
    .reset_index(drop=True)
)

print("Contextos WVS:", len(contextos))
print("Contextos eleitorais:", len(eleicoes))

display(contextos.head())
display(eleicoes.head())

Contextos WVS: 26
Contextos eleitorais: 437


,country_text_id,year_wvs
0,ARG,2013
1,ARG,2017
2,BOL,2017
3,BRA,2014
4,BRA,2018


,country_text_id,year_eleicao
0,ARG,1912
1,ARG,1914
2,ARG,1916
3,ARG,1918
4,ARG,1920


In [38]:
# ============================================================
# Harmonização temporal país a país
# ============================================================

contextos_harmonizados = []

for pais in sorted(contextos["country_text_id"].unique()):

    wvs_pais = (
        contextos
        .loc[contextos["country_text_id"] == pais]
        .sort_values("year_wvs")
    )

    eleicoes_pais = (
        eleicoes
        .loc[eleicoes["country_text_id"] == pais]
        .sort_values("year_eleicao")
    )

    # País sem informações no V-Party
    if eleicoes_pais.empty:
        wvs_pais = wvs_pais.copy()
        wvs_pais["year_eleicao"] = np.nan
        contextos_harmonizados.append(wvs_pais)
        continue

    contexto = pd.merge_asof(
        wvs_pais,
        eleicoes_pais,
        left_on="year_wvs",
        right_on="year_eleicao",
        direction="backward"
    )

    contextos_harmonizados.append(contexto)

contexto_vparty = (
    pd.concat(contextos_harmonizados, ignore_index=True)
)

# Corrige colunas geradas pelo merge_asof
contexto_vparty["country_text_id"] = (
    contexto_vparty["country_text_id_x"]
    .fillna(contexto_vparty["country_text_id_y"])
)

contexto_vparty = (
    contexto_vparty[
        ["country_text_id", "year_wvs", "year_eleicao"]
    ]
    .sort_values(["country_text_id", "year_wvs"])
    .reset_index(drop=True)
)

display(contexto_vparty)

,country_text_id,year_wvs,year_eleicao
0,ARG,2013,2013.0
1,ARG,2017,2017.0
2,BOL,2017,2014.0
3,BRA,2014,2014.0
4,BRA,2018,2018.0
5,CAN,2020,2019.0
6,CHL,2012,2009.0
7,CHL,2018,2017.0
8,COL,2012,2010.0
9,COL,2018,2018.0


In [39]:
# Remove contextos sem correspondência no V-Party
contexto_vparty = (
    contexto_vparty
    .dropna(subset=["country_text_id", "year_eleicao"])
    .copy()
)

contexto_vparty["year_eleicao"] = contexto_vparty["year_eleicao"].astype(int)

print(contexto_vparty.shape)
display(contexto_vparty)

(25, 3)


,country_text_id,year_wvs,year_eleicao
0,ARG,2013,2013
1,ARG,2017,2017
2,BOL,2017,2014
3,BRA,2014,2014
4,BRA,2018,2018
5,CAN,2020,2019
6,CHL,2012,2009
7,CHL,2018,2017
8,COL,2012,2010
9,COL,2018,2018


In [40]:
# ============================================================
# Integração dos contextos WVS com indicadores do V-Party
# ============================================================

vparty_contexto = (
    contexto_vparty
    .merge(
        vparty_agregado,
        left_on=["country_text_id", "year_eleicao"],
        right_on=["country_text_id", "year"],
        how="left"
    )
    .drop(columns=["year"])
)

print(vparty_contexto.shape)

display(vparty_contexto.head())

(25, 60)


,country_text_id,year_wvs,year_eleicao,n_partidos,numero_efetivo_partidos,concentracao_partidaria,populismo_ponderado,populismo_max,populismo_sd,populismo_range,...,polarizacao_ideologica,heterogeneidade_populista,cobertura_populismo,cobertura_antipluralismo,cobertura_ideologia,cobertura_organizacao,cobertura_clientelismo,n_partidos_sistema,n_partidos_com_populismo,n_partidos_com_ideologia
0,ARG,2013,2013,4.0,2.390812,0.586449,0.667145,0.872,0.307045,0.734,...,0.984925,0.256768,1.000000,1.000000,1.000000,1.000000,1.000000,4.0,4.0,4.0
1,ARG,2017,2017,8.0,5.266373,0.312173,0.403522,0.670,0.184826,0.525,...,1.198420,0.210043,0.645258,0.645258,0.645258,0.645258,0.645258,8.0,6.0,6.0
2,BOL,2017,2014,3.0,1.224569,0.897878,0.825110,0.892,0.372525,0.655,...,1.099196,0.198340,1.000000,1.000000,1.000000,1.000000,1.000000,3.0,3.0,3.0
3,BRA,2014,2014,11.0,9.189256,0.163995,0.307455,0.688,0.185379,0.590,...,1.346485,0.197370,1.000000,1.000000,1.000000,1.000000,1.000000,11.0,11.0,11.0
4,BRA,2018,2018,11.0,9.369120,0.155492,0.341291,0.777,0.211666,0.679,...,1.798684,0.228993,1.000000,1.000000,1.000000,1.000000,1.000000,11.0,11.0,11.0


In [41]:
# ============================================================
# Validação do contexto V-Party
# ============================================================

print("Contextos WVS:", contexto_vparty.shape[0])
print("Contextos V-Party integrados:", vparty_contexto.shape[0])

print("\nCobertura populismo:")
display(vparty_contexto["cobertura_populismo"].describe())

print("\nContextos com cobertura populismo < 0.9:")
display(
    vparty_contexto.loc[
        vparty_contexto["cobertura_populismo"] < 0.9,
        ["country_text_id", "year_wvs", "year_eleicao",
         "cobertura_populismo", "n_partidos_sistema",
         "n_partidos_com_populismo"]
    ]
)

Contextos WVS: 25
Contextos V-Party integrados: 25

Cobertura populismo:


count    25.000000
mean      0.959696
std       0.129257
min       0.447039
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: cobertura_populismo, dtype: float64


Contextos com cobertura populismo < 0.9:


,country_text_id,year_wvs,year_eleicao,cobertura_populismo,n_partidos_sistema,n_partidos_com_populismo
1,ARG,2017,2017,0.645258,8.0,6.0
24,VEN,2021,2015,0.447039,7.0,5.0


In [42]:
# ============================================================
# Persistência do contexto V-Party harmonizado
# ============================================================

ARQ_VPARTY_CONTEXTO = DIR_INTEGRADOS / "vparty_contexto.parquet"

vparty_contexto.to_parquet(
    ARQ_VPARTY_CONTEXTO,
    index=False
)

print("Arquivo salvo em:")
print(ARQ_VPARTY_CONTEXTO)

Arquivo salvo em:
C:\Users\tmaci\Documents\Doutorado_Populismo\dados\integrados\vparty_contexto.parquet


In [43]:
# ============================================================
# Integração final do V-Party à base multinível
# ============================================================

base_analitica = (
    base
    .merge(
        vparty_contexto.drop(columns=["year_wvs"]),
        left_on=["country_text_id", "year"],
        right_on=["country_text_id", "year_eleicao"],
        how="left"
    )
)

print("Base original :", base.shape)
print("Base analítica:", base_analitica.shape)

display(base_analitica.head())

Base original : (39720, 53)
Base analítica: (39720, 111)


,country_text_id,year,national_pride,willing_to_fight,grandiosity,reject_different_race,reject_immigrants,reject_homosexuals,reject_other_religion,reject_different_language,...,polarizacao_ideologica,heterogeneidade_populista,cobertura_populismo,cobertura_antipluralismo,cobertura_ideologia,cobertura_organizacao,cobertura_clientelismo,n_partidos_sistema,n_partidos_com_populismo,n_partidos_com_ideologia
0,ARG,2013,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.984925,0.256768,1.0,1.0,1.0,1.0,1.0,4.0,4.0,4.0
1,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.984925,0.256768,1.0,1.0,1.0,1.0,1.0,4.0,4.0,4.0
2,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.984925,0.256768,1.0,1.0,1.0,1.0,1.0,4.0,4.0,4.0
3,ARG,2013,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.984925,0.256768,1.0,1.0,1.0,1.0,1.0,4.0,4.0,4.0
4,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.984925,0.256768,1.0,1.0,1.0,1.0,1.0,4.0,4.0,4.0


In [44]:
# ============================================================
# Auditoria da integração final
# ============================================================

print("Observações:")
print(f"Base original  : {len(base):,}")
print(f"Base analítica : {len(base_analitica):,}")

assert len(base) == len(base_analitica)

print("\nPaíses:")
print(base_analitica["country_text_id"].nunique())

print("\nPeríodo:")
print(
    base_analitica["year"].min(),
    "-",
    base_analitica["year"].max()
)

print("\nCobertura do V-Party:")

print(
    base_analitica[
        "numero_efetivo_partidos"
    ].notna().mean()
)

Observações:
Base original  : 39,720
Base analítica : 39,720

Países:
17

Período:
2010 - 2022

Cobertura do V-Party:
0.3208207452165156


In [45]:
# ============================================================
# Persistência da base analítica
# ============================================================

ARQ_BASE_ANALITICA = DIR_INTEGRADOS / "base_analitica.parquet"

base_analitica.to_parquet(
    ARQ_BASE_ANALITICA,
    index=False
)

print("Arquivo salvo em:")
print(ARQ_BASE_ANALITICA)

Arquivo salvo em:
C:\Users\tmaci\Documents\Doutorado_Populismo\dados\integrados\base_analitica.parquet


## Resumo da etapa



# Conclusões do Notebook

## Objetivo alcançado

Este notebook teve como finalidade incorporar à base multinível do projeto as características do sistema partidário provenientes do banco **V-Dem CPD Party Dataset (V-Party)**, produzindo uma base analítica integrada entre atitudes individuais, contexto institucional e contexto partidário.

Ao final desta etapa foi gerada uma nova base analítica contendo, simultaneamente:

- variáveis individuais derivadas do World Values Survey (WVS);
- fatores latentes obtidos nos experimentos anteriores;
- indicadores institucionais provenientes do V-Dem;
- indicadores agregados do sistema partidário provenientes do V-Party.

Essa integração constitui a base definitiva para as análises relacionais previstas nas etapas subsequentes da pesquisa.

---

## Preparação dos dados

Inicialmente foi realizada a leitura do banco mundial do V-Party, preservando sua estrutura original.

Em seguida, optou-se por restringir todas as análises ao universo efetivamente utilizado na tese, isto é, aos países presentes na base multinível construída anteriormente a partir da integração entre WVS e V-Dem.

Essa decisão metodológica reduz processamento desnecessário, elimina observações irrelevantes ao desenho empírico da pesquisa e mantém uma versão mundial preservada para reutilização em estudos futuros.

---

## Construção dos indicadores do sistema partidário

A partir dos microdados partidários foram produzidos indicadores agregados no nível **país-eleição**, sintetizando propriedades estruturais dos sistemas partidários.

Foram calculados indicadores relativos à estrutura competitiva do sistema, incluindo:

- número de partidos;
- número efetivo de partidos (Laakso & Taagepera);
- concentração partidária.

Esses indicadores permitem caracterizar a fragmentação e o grau de concentração eleitoral de cada sistema político.

---

## Indicadores substantivos do campo partidário

Foram posteriormente agregadas diversas dimensões substantivas disponíveis no V-Party.

Para cada dimensão foram calculados:

- média ponderada pelo peso eleitoral/parlamentar dos partidos;
- valor máximo observado;
- desvio-padrão;
- amplitude (range).

As dimensões incorporadas compreendem:

- populismo;
- antipluralismo;
- antielitismo;
- people-centrism;
- pluralismo;
- culto à liderança;
- organização partidária;
- capilaridade organizacional;
- atuação oposicionista;
- clientelismo;
- posicionamento ideológico esquerda-direita.

Esses indicadores descrevem o contexto partidário enfrentado pelos respondentes do WVS em cada país.

---

## Indicadores de heterogeneidade

Além das estatísticas tradicionais, foram produzidos indicadores destinados a mensurar a heterogeneidade interna dos sistemas partidários.

Em particular foram implementados:

- polarização ideológica ponderada;
- heterogeneidade do populismo.

Esses indicadores utilizam desvios ponderados pelo peso eleitoral dos partidos, evitando que pequenas legendas produzam artificialmente elevados níveis de polarização.

Tal procedimento fornece uma medida mais substantiva da distribuição efetiva das posições partidárias.

---

## Diagnóstico de cobertura

Foi igualmente implementado um sistema de avaliação da qualidade da informação disponível em cada contexto político.

Para cada país-eleição foram calculadas medidas de cobertura relativas às principais dimensões analíticas, bem como o número de partidos efetivamente codificados.

Esses indicadores permitem identificar contextos parcialmente observados e poderão ser utilizados posteriormente em análises de sensibilidade e testes de robustez.

---

## Harmonização temporal

Uma etapa metodológica adicional tornou-se necessária em razão da natureza distinta das bases utilizadas.

Enquanto o World Values Survey registra o **ano da entrevista**, o V-Party organiza suas informações segundo o **ano da eleição**.

Uma integração direta por igualdade de anos produziria elevada perda de informação e introduziria inconsistências temporais.

Para solucionar esse problema foi implementado um procedimento de harmonização temporal baseado na eleição imediatamente anterior (ou coincidente) ao ano da entrevista.

Assim, cada respondente do WVS passou a ser associado ao contexto partidário efetivamente vigente no momento em que respondeu ao questionário, preservando a precedência temporal entre contexto político e atitudes individuais.

Essa estratégia segue práticas consolidadas na literatura comparada e evita o uso de informações provenientes de eleições futuras.

---

## Integração multinível

Após a harmonização temporal, os indicadores agregados do sistema partidário foram incorporados à base multinível existente.

Cada observação individual passou a conter simultaneamente:

- atributos individuais;
- fatores latentes;
- contexto institucional;
- contexto partidário.

Essa estrutura permitirá estimar modelos multinível, análises fatoriais relacionais, análises geométricas (MCA, MFA e CA), técnicas de agrupamento e demais estratégias empíricas previstas na tese.

---

## Produtos gerados

Ao término deste notebook foram produzidos os seguintes arquivos:

- `vparty_americas.parquet`
- `vparty_contexto.parquet`
- `base_analitica.parquet`

Esses arquivos constituem a infraestrutura analítica definitiva utilizada nas etapas subsequentes da pesquisa.

---

## Próximas etapas

A partir da base analítica integrada, os próximos experimentos dedicar-se-ão à exploração das relações entre:

- dimensões individuais do narcisismo coletivo;
- configurações institucionais;
- características dos sistemas partidários;
- contextos nacionais.

Essas análises utilizarão técnicas relacionais, particularmente Multiple Correspondence Analysis (MCA), Multiple Factor Analysis (MFA) e Correspondence Analysis (CA), permitindo investigar como diferentes configurações institucionais e partidárias estruturam os perfis contemporâneos do populismo e do narcisismo coletivo.

Nota de rodapé

A harmonização temporal pelo critério da "eleição imediatamente anterior ou coincidente" não é apenas uma solução técnica, mas uma decisão metodológica explícita da tese. Vale a pena registrá-la porque ela justifica cientificamente a integração entre o WVS e o V-Party e poderá ser reproduzida ou defendida na descrição metodológica da tese e em eventuais artigos derivados do projeto.

## 4. Exportação dos resultados

In [48]:
# ============================================================
# Exportação dos produtos do Notebook
# ============================================================

ARQ_VPARTY_AMERICAS = DIR_INTEGRADOS / "vparty_americas.parquet"
ARQ_VPARTY_AGREGADO = DIR_INTEGRADOS / "vparty_agregado.parquet"
ARQ_VPARTY_CONTEXTO = DIR_INTEGRADOS / "vparty_contexto.parquet"
ARQ_BASE_ANALITICA = DIR_INTEGRADOS / "base_analitica.parquet"

vparty.to_parquet(
    ARQ_VPARTY_AMERICAS,
    index=False
)

vparty_agregado.to_parquet(
    ARQ_VPARTY_AGREGADO,
    index=False
)

vparty_contexto.to_parquet(
    ARQ_VPARTY_CONTEXTO,
    index=False
)

base_analitica.to_parquet(
    ARQ_BASE_ANALITICA,
    index=False
)

print("Arquivos exportados com sucesso.\n")

for arquivo in [
    ARQ_VPARTY_AMERICAS,
    ARQ_VPARTY_AGREGADO,
    ARQ_VPARTY_CONTEXTO,
    ARQ_BASE_ANALITICA,
]:
    print(f"✓ {arquivo.name}")

Arquivos exportados com sucesso.

✓ vparty_americas.parquet
✓ vparty_agregado.parquet
✓ vparty_contexto.parquet
✓ base_analitica.parquet


In [49]:
# ============================================================
# Relatório final
# ============================================================

print("=" * 70)
print("EXPERIMENTO 07 — INTEGRAÇÃO V-PARTY")
print("=" * 70)

print(f"\nObservações WVS: {len(base_analitica):,}")
print(f"Países: {base_analitica['country_text_id'].nunique()}")
print(f"Período: {base_analitica['year'].min()}–{base_analitica['year'].max()}")

print("\nProdutos gerados:")

produtos = {
    "vparty_americas": vparty,
    "vparty_agregado": vparty_agregado,
    "vparty_contexto": vparty_contexto,
    "base_analitica": base_analitica,
}

for nome, df in produtos.items():
    print(f"{nome:20} {df.shape[0]:>8,} linhas × {df.shape[1]} colunas")

print("\nCobertura V-Party:")

cobertura = (
    base_analitica["numero_efetivo_partidos"]
    .notna()
    .mean()
)

print(f"{100*cobertura:.1f}% das observações receberam contexto partidário.")

print("\nNotebook concluído com sucesso.")

EXPERIMENTO 07 — INTEGRAÇÃO V-PARTY

Observações WVS: 39,720
Países: 17
Período: 2010–2022

Produtos gerados:
vparty_americas         1,638 linhas × 30 colunas
vparty_agregado           437 linhas × 59 colunas
vparty_contexto            25 linhas × 60 colunas
base_analitica         39,720 linhas × 111 colunas

Cobertura V-Party:
32.1% das observações receberam contexto partidário.

Notebook concluído com sucesso.


# Organização dos produtos analíticos

## Objetivo

A etapa final deste notebook consolida os produtos gerados durante a integração do V-Party e estabelece sua organização dentro da estrutura permanente do projeto.

Optou-se por preservar todos os produtos intermediários produzidos ao longo do pipeline analítico, evitando que futuras atualizações das bases originais exijam a reconstrução completa de todas as etapas já executadas.

Essa estratégia também favorece a reprodutibilidade da pesquisa, permitindo que cada transformação dos dados permaneça explicitamente documentada e possa ser auditada de forma independente.

---

## Estrutura dos produtos

Os arquivos produzidos neste notebook possuem funções distintas dentro do pipeline analítico.

### Produtos intermediários

Os arquivos intermediários representam transformações sucessivas do banco V-Party e poderão ser reutilizados em experimentos futuros sem necessidade de nova preparação dos dados.

Esses produtos compreendem:

- `vparty_americas.parquet`: recorte geográfico do V-Party correspondente ao universo analítico da tese;

- `vparty_agregado.parquet`: indicadores agregados do sistema partidário calculados no nível país-eleição;

- `vparty_contexto.parquet`: harmonização temporal entre os anos das entrevistas do World Values Survey e os anos eleitorais do V-Party.

---

### Produto analítico final

O arquivo

`base_analitica.parquet`

constitui a principal saída deste notebook.

Nele encontram-se integrados:

- dados individuais do World Values Survey;
- fatores latentes produzidos nos experimentos anteriores;
- indicadores institucionais do V-Dem;
- indicadores do sistema partidário provenientes do V-Party.

Essa base passa a constituir a referência oficial para todas as etapas empíricas subsequentes da tese.

---

## Arquitetura do pipeline

Após a conclusão deste notebook, o fluxo de dados do projeto passa a assumir a seguinte estrutura lógica:

```text
WVS
        │
        ├───────────────┐
        │               │
        ▼               ▼
 Experimentos      Integração V-Dem
        │               │
        └──────┬────────┘
               │
               ▼
      base_multinivel.parquet
               │
               ▼
      Integração V-Party
               │
               ├──────────────┐
               │              │
               ▼              ▼
     vparty_agregado   vparty_contexto
               │              │
               └──────┬───────┘
                      │
                      ▼
          base_analitica.parquet
                      │
                      ▼
        Experimentos relacionais
     (MCA • MFA • CA • Modelos multinível)
```

---

## Considerações metodológicas

A preservação explícita dos produtos intermediários constitui uma decisão metodológica da pesquisa.

Além de facilitar auditorias e testes de robustez, essa organização permite que futuras atualizações das bases V-Dem, V-Party ou WVS sejam incorporadas ao projeto mediante a reconstrução apenas das etapas efetivamente afetadas, preservando os demais componentes do pipeline.

Dessa forma, o projeto mantém simultaneamente reprodutibilidade, modularidade e rastreabilidade de todas as transformações realizadas ao longo do processo analítico.

# Validação e persistência dos produtos

## Objetivo

A etapa final deste notebook realiza a validação da base analítica produzida e registra definitivamente todos os produtos gerados durante a integração do V-Party.

A validação possui três objetivos complementares.

Primeiramente, verificar se nenhuma observação foi perdida, duplicada ou reordenada ao longo das sucessivas integrações realizadas neste notebook.

Em seguida, avaliar a cobertura efetiva dos indicadores partidários incorporados à base multinível, identificando eventuais lacunas que poderão ser consideradas em análises de sensibilidade.

Por fim, persistir todos os produtos intermediários e a base analítica definitiva, garantindo a reprodutibilidade do pipeline e permitindo que as etapas subsequentes da pesquisa utilizem diretamente os dados consolidados, sem necessidade de reconstrução das transformações aqui implementadas.

Ao término desta etapa considera-se concluída a integração entre World Values Survey, V-Dem e V-Party, encerrando a fase de preparação da infraestrutura empírica da tese.

In [51]:
# ============================================================
# Auditoria da integração final
# ============================================================

print("=" * 70)
print("AUDITORIA FINAL")
print("=" * 70)

print("\nBase multinível")
print(f"Observações : {len(base):,}")
print(f"Variáveis   : {base.shape[1]}")

print("\nBase analítica")
print(f"Observações : {len(base_analitica):,}")
print(f"Variáveis   : {base_analitica.shape[1]}")

assert len(base) == len(base_analitica), \
    "Erro: houve alteração no número de observações."

assert base.index.equals(base_analitica.index), \
    "Erro: a ordem das observações foi modificada."

print("\n✓ Estrutura preservada.")

print("\nCobertura dos principais indicadores:")

indicadores = [
    "numero_efetivo_partidos",
    "concentracao_partidaria",
    "populismo_ponderado",
    "polarizacao_ideologica",
]

for indicador in indicadores:

    cobertura = (
        base_analitica[indicador]
        .notna()
        .mean()
    )

    print(f"{indicador:<35} {100*cobertura:6.2f}%")

AUDITORIA FINAL

Base multinível
Observações : 39,720
Variáveis   : 53

Base analítica
Observações : 39,720
Variáveis   : 111

✓ Estrutura preservada.

Cobertura dos principais indicadores:
numero_efetivo_partidos              32.08%
concentracao_partidaria              32.08%
populismo_ponderado                  32.08%
polarizacao_ideologica               32.08%


In [52]:
# ============================================================
# Exportação dos produtos
# ============================================================

produtos = {
    "vparty_americas.parquet": vparty,
    "vparty_agregado.parquet": vparty_agregado,
    "vparty_contexto.parquet": vparty_contexto,
    "base_analitica.parquet": base_analitica,
}

print("=" * 70)
print("EXPORTAÇÃO")
print("=" * 70)

for nome, dataframe in produtos.items():

    caminho = DIR_INTEGRADOS / nome

    dataframe.to_parquet(
        caminho,
        index=False
    )

    print(f"✓ {nome}")

print("\nTodos os arquivos foram exportados com sucesso.")

EXPORTAÇÃO
✓ vparty_americas.parquet
✓ vparty_agregado.parquet
✓ vparty_contexto.parquet
✓ base_analitica.parquet

Todos os arquivos foram exportados com sucesso.


In [56]:
# ============================================================
# Recarrega módulos do projeto durante o desenvolvimento
# ============================================================

import importlib
import project_tools

importlib.reload(project_tools)

from project_tools import *

In [57]:
relatorio_notebook(
    nome="Experimento 07 — Integração V-Party",
    base=base_analitica,
    produtos=produtos,
    proxima_etapa="Experimento 08 — Análises relacionais do campo político."
)

EXPERIMENTO 07 — INTEGRAÇÃO V-PARTY

Produtos gerados
vparty_americas.parquet        1,638 linhas × 30 colunas
vparty_agregado.parquet          437 linhas × 59 colunas
vparty_contexto.parquet           25 linhas × 60 colunas
base_analitica.parquet        39,720 linhas × 111 colunas

Base analítica
Observações : 39,720
Variáveis   : 111
Países      : 17
Período     : 2010–2022

Notebook concluído com sucesso.

Próxima etapa:
Experimento 08 — Análises relacionais do campo político.


# Auditoria da infraestrutura do projeto

## Objetivo

Antes de encerrar a etapa de integração entre o World Values Survey, o Varieties of Democracy e o V-Party, realiza-se uma auditoria da infraestrutura analítica do projeto.

Essa auditoria possui quatro objetivos principais:

- verificar a existência dos principais arquivos produzidos ao longo do pipeline;
- confirmar a consistência estrutural da base analítica definitiva;
- validar a disponibilidade dos produtos intermediários necessários à reprodutibilidade da pesquisa;
- registrar o encerramento da fase de construção da infraestrutura de dados.

A aprovação desta auditoria indica que a pesquisa encontra-se pronta para iniciar a etapa de análises relacionais.

In [61]:
# ============================================================
# Auditoria dos arquivos do projeto
# ============================================================

arquivos = [
    DIR_INTEGRADOS / "base_multinivel.parquet",
    DIR_INTEGRADOS / "vparty_americas.parquet",
    DIR_INTEGRADOS / "vparty_agregado.parquet",
    DIR_INTEGRADOS / "vparty_contexto.parquet",
    DIR_INTEGRADOS / "base_analitica.parquet",
]

print("=" * 70)
print("AUDITORIA DOS ARQUIVOS")
print("=" * 70)

faltando = []

for arq in arquivos:
    if arq.exists():
        print(f"✓ {arq.name}")
    else:
        print(f"✗ {arq.name}")
        faltando.append(arq.name)

print()

if faltando:
    print("Arquivos ausentes:")
    for arq in faltando:
        print(f" - {arq}")
else:
    print("Todos os arquivos esperados foram encontrados.")

AUDITORIA DOS ARQUIVOS
✓ base_multinivel.parquet
✓ vparty_americas.parquet
✓ vparty_agregado.parquet
✓ vparty_contexto.parquet
✓ base_analitica.parquet

Todos os arquivos esperados foram encontrados.
